In [ ]:
r0 = float(r_clipped[0])

cir_curve = CIRCurve(
    short_rate=r0,
    theta=theta_hat,
    kappa=kappa_hat,
    beta=beta_hat
)
# was after "residuals from AR1

In [ ]:
# Tenors/maturities in years
tenors = np.array([0.25] + list(range(1, 11)))

# pick a few time indices from term_structure
idx_first  = term_structure.index[0]
idx_middle = term_structure.index[len(term_structure)//2]
idx_last   = term_structure.index[-1]
date_indices = [idx_first, idx_middle, idx_last]

fig, ax = plt.subplots(figsize=(8,5))

for idx in date_indices:
    # observed bootstrapped yields for that date
    y_obs = term_structure.loc[idx, ["y_0_25"] + [f"y_{n}Y" for n in range(1,11)]].values

    # current short rate at that date (3M)
    r_t = float(term_structure.loc[idx, "y_0_25"])

    # build CIR curve at that time
    cir_curve_t = CIRCurve(
        short_rate=r_t,
        theta=theta_hat,
        kappa=kappa_hat,
        beta=beta_hat
    )

    # model-implied yields for all tenors
    y_model = cir_curve_t.zero_rate_vector(tenors)

    # plot
    plt.plot(tenors, y_obs,   marker="o", linestyle="-",  label=f"Obs, t={idx}")
    plt.plot(tenors, y_model, marker="x", linestyle="--", label=f"CIR, t={idx}")

plt.xlabel("Maturity (years)")
plt.ylabel("Yield (cont. comp.)")
plt.title("Bootstrapped vs CIR Model Yields")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()

# One or 2 factor sanity check not usefull. comes after eps_hat

In [ ]:
# Comes after first two dataclasses.

rng = np.random.default_rng()

def simulate_chen_scott_2f(params: ChenScott2FParams,
                           y0_1: float,
                           y0_2: float,
                           n_steps: int,
                           dt: float = DT):
    """
    Simulate 2-factor Chen–Scott model.
    Returns factor1, factor2 and short-rate r_t = y1 + y2.
    """
    y1 = np.zeros(n_steps + 1)
    y2 = np.zeros(n_steps + 1)
    r  = np.zeros(n_steps + 1)

    y1[0] = max(y0_1, 1e-8)
    y2[0] = max(y0_2, 1e-8)
    r[0]  = y1[0] + y2[0]

    rho = np.clip(params.rho, -0.999, 0.999)

    for t in range(n_steps):
        # correlated normals
        z1, z2 = rng.normal(), rng.normal()
        e1 = z1
        e2 = rho * z1 + np.sqrt(1 - rho**2) * z2

        # factor 1
        f1 = params.factor1
        y1_pos = max(y1[t], 1e-8)
        drift1 = f1.kappa * (f1.theta - y1_pos) * dt
        diff1  = f1.sigma * np.sqrt(y1_pos) * np.sqrt(dt) * e1
        y1[t+1] = max(y1_pos + drift1 + diff1, 1e-8)

        # factor 2
        f2 = params.factor2
        y2_pos = max(y2[t], 1e-8)
        drift2 = f2.kappa * (f2.theta - y2_pos) * dt
        diff2  = f2.sigma * np.sqrt(y2_pos) * np.sqrt(dt) * e2
        y2[t+1] = max(y2_pos + drift2 + diff2, 1e-8)

        r[t+1] = y1[t+1] + y2[t+1]

    return y1, y2, r

In [ ]:
"""Old montecarlo sim of short rate process with non tuned factors. """

# 0. Settings for the simulation
# ------------------------------
num_sims = 10_000        # number of Monte Carlo scenarios
n_steps  = 52            # 1-year horizon with weekly steps
dt       = 1.0 / 52.0    # time step in years
seed     = 12345         # for reproducibility (optional)

# -------------------------------------------
# 1. Re-estimate CIR params for short & long
# -------------------------------------------
k1, th1, s1 = estimate_cir_from_series(term_structure["y_0_25"], dt)
k2, th2, s2 = estimate_cir_from_series(term_structure["y_10Y"], dt)

factor1_params = CIRFactorQ(kappa=k1, theta=th1, sigma=s1)  # short-rate factor
factor2_params = CIRFactorQ(kappa=k2, theta=th2, sigma=s2)  # long-rate factor

# -------------------------------------------
# 2. Initial factor levels at "today" (t = 0)
# -------------------------------------------
today_idx = term_structure.index[-1]
y1_0 = float(term_structure.loc[today_idx, "y_0_25"])  # starting short end
y2_0 = float(term_structure.loc[today_idx, "y_10Y"])   # starting long end

# -------------------------------------------
# 3. Simulate 2-factor CIR (Chen–Scott) paths
# -------------------------------------------
rng = np.random.default_rng(seed)

# allocate arrays: shape (num_sims, n_steps+1)
y1_paths = np.zeros((num_sims, n_steps + 1))
y2_paths = np.zeros((num_sims, n_steps + 1))
r_paths  = np.zeros((num_sims, n_steps + 1))  # short rate = y1 + y2

# set initial values
y1_paths[:, 0] = max(y1_0, 1e-8)
y2_paths[:, 0] = max(y2_0, 1e-8)
r_paths[:, 0]  = y1_paths[:, 0] + y2_paths[:, 0]

rho = 0.0  # factor correlation (keep 0 for simplicity)

for t in range(n_steps):
    # correlated normal shocks for all scenarios at step t
    z1 = rng.normal(size=num_sims)
    z2 = rng.normal(size=num_sims)
    e1 = z1
    e2 = rho * z1 + np.sqrt(1 - rho**2) * z2

    # factor 1 update
    y1_t = np.clip(y1_paths[:, t], 1e-8, None)
    drift1 = factor1_params.kappa * (factor1_params.theta - y1_t) * dt
    diff1  = factor1_params.sigma * np.sqrt(y1_t) * np.sqrt(dt) * e1
    y1_next = np.clip(y1_t + drift1 + diff1, 1e-8, None)
    y1_paths[:, t+1] = y1_next

    # factor 2 update
    y2_t = np.clip(y2_paths[:, t], 1e-8, None)
    drift2 = factor2_params.kappa * (factor2_params.theta - y2_t) * dt
    diff2  = factor2_params.sigma * np.sqrt(y2_t) * np.sqrt(dt) * e2
    y2_next = np.clip(y2_t + drift2 + diff2, 1e-8, None)
    y2_paths[:, t+1] = y2_next

    # short rate
    r_paths[:, t+1] = y1_next + y2_next

# quick sanity check
print("y1_paths shape:", y1_paths.shape)
print("y2_paths shape:", y2_paths.shape)
print("r_paths shape: ", r_paths.shape)
print("Example first-path short rates (first few steps):")
print(r_paths[0, :5])

In [ ]:
num_sims, n_steps_plus1 = y1_paths.shape
assert n_steps_plus1 == 53  # 0..52 weeks

# ------------------------------
# 1. Factor values at t = 1 year
# ------------------------------
# We take the last column (week 52) as "1 year"
y1_1year = y1_paths[:, -1]  # shape (num_sims,)
y2_1year = y2_paths[:, -1]

# ------------------------------
# 2. Settings for bonds
# ------------------------------
maturities = np.arange(1, 11)   # 1..10 years
coupon_rate = 0.06
principal = 100.0
annual_coupon = coupon_rate * principal  # 0.06

# We will store bond prices at t = 1 in an array (num_sims x 10)
bond_prices_1 = np.zeros((num_sims, len(maturities)))

# ------------------------------
# 3. Loop over scenarios and price bonds at t = 1
# ------------------------------
for i in range(num_sims):
    # Build the 2-factor curve for scenario i at t = 1 year
    curve_i = TwoFactorCIRCurve(
        y1=float(y1_1year[i]),
        y2=float(y2_1year[i]),
        factor1=factor1_params,
        factor2=factor2_params
    )

    # For pricing from t = 1, the remaining maturities go up to 9 years (10Y bond has 9 years left)
    remaining_tenors = np.arange(1, 10)  # 1..9 years
    # Corresponding zero rates and discount factors under the model
    y_rem = curve_i.zero_rate_vector(remaining_tenors)
    df_rem = np.exp(-y_rem * remaining_tenors)

    # Price each bond
    for j, n in enumerate(maturities):
        if n == 1:
            # The 1Y bond matures exactly at t = 1 year:
            # market value after paying principal is zero; price_1 = 0
            bond_prices_1[i, j] = 0.0
        else:
            # From t = 1 perspective, there are (n-1) years left.
            # Cash flows: coupon each year, principal at final maturity.
            n_rem = n - 1
            cfs_rem = np.full(n_rem, annual_coupon)
            cfs_rem[-1] += principal  # add principal at final remaining maturity

            # Use discount factors for tenors 1..(n-1)
            dfs_n = df_rem[:n_rem]

            price_1 = np.sum(cfs_rem * dfs_n)
            bond_prices_1[i, j] = price_1

# Quick sanity check: shapes and a few example prices
print("bond_prices_1 shape:", bond_prices_1.shape)
print("Example scenario 0 prices at t=1 (1Y..10Y):")
print(bond_prices_1[0])

In [ ]:
num_sims, n_steps_plus1 = r_paths.shape
dt = 1.0 / 52.0

# ------------------------------
# 1. Bank account 1-year return
# ------------------------------
# Approximate integral of short rate over [0,1]:
# use rates at steps 0..51 (52 intervals)
r_for_integral = r_paths[:, :-1]  # shape (num_sims, 52)

# log growth = ∑ r_t * dt  ->  B_1 = exp(log_growth)
log_growth = np.sum(r_for_integral * dt, axis=1)
bank_account_1 = np.exp(log_growth)

# Since bank_account_0 = 1, simple return is:
bank_returns = bank_account_1 - 1.0   # shape (num_sims,)

# ------------------------------
# 2. Bond 1-year returns
# ------------------------------
maturities = np.arange(1, 11)     # 1..10 years
coupon_rate = 0.06
principal = 100.0
annual_coupon = coupon_rate * principal  # 0.06

# Initial prices in correct order (1Y..10Y)
P0_bonds = np.array([bond_prices_0[f"bond_{n}Y"] for n in maturities])  # shape (10,)

# Cash flows at t=1:
# - 1Y bond: coupon + principal
# - n>1:     coupon only (principal comes later, already in P1)
coupon_cf = np.full_like(P0_bonds, annual_coupon, dtype=float)
coupon_cf[0] += principal  # add principal for 1Y bond

# Payoff per scenario and bond at t=1:
# payoff = price at t=1 + cash flow at t=1
# bond_prices_1 has shape (num_sims, 10)
payoff_bonds_1 = bond_prices_1 + coupon_cf  # broadcasting, shape (num_sims, 10)

# Simple returns: R = payoff / P0 - 1
bond_returns = payoff_bonds_1 / P0_bonds - 1.0   # shape (num_sims, 10)

# ------------------------------
# 3. Collect all fixed-income returns
# ------------------------------
# Build a DataFrame with all scenarios
fi_returns = pd.DataFrame(
    np.column_stack([bank_returns, bond_returns]),
    columns=["Bank_account"] + [f"bond_{n}Y" for n in maturities]
)

print(fi_returns.head())
print("\nMeans of 1-year returns (bank + bonds):")
print(fi_returns.mean())
print("\nStd dev of 1-year returns:")
print(fi_returns.std())


In [ ]:
""" old boxplot of 1y FI_asset returns."""

fig, ax = plt.subplots(figsize=(12, 6))

# Select all fixed-income assets (bank + bonds)
fi_assets = ["Bank_account"] + [f"bond_{n}Y" for n in range(1, 11)]

# Long-form dataframe for seaborn
fi_returns_melted = (
    fi_returns[fi_assets]
    .melt(var_name="Asset", value_name="Return")
)

# Create the boxplot
sns.boxplot(
    data=fi_returns_melted,
    x="Asset",
    y="Return",
    palette="viridis"
)

plt.title("Distribution of 1-Year Returns Across Fixed-Income Assets")
plt.ylabel("1-Year Return")
plt.xlabel("")
plt.grid(axis='y', linestyle="--", alpha=0.4)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
""" old plot of mcauly duration vs. expected returns of bonds (still referes to fi_returns"""

coupon_rate = 0.06
principal = 100.0
annual_coupon = coupon_rate * principal

maturities = np.arange(1, 11)

# Extract today's yields from your bootstrapped term structure
y_today = term_structure.loc[term_structure.index[-1]]
spot_yields_today = np.array([y_today[f"y_{n}Y"] for n in maturities])
dfs_today = np.exp(-spot_yields_today * maturities)

durations = []

for n in maturities:
    # Cash flows: coupon each year, plus principal at maturity
    cfs = np.full(n, annual_coupon)
    cfs[-1] += principal

    # Discount factors for each cash flow date
    df_n = np.exp(-spot_yields_today[:n] * np.arange(1, n+1))

    # Present value
    pv = np.sum(cfs * df_n)

    # Macaulay duration
    times = np.arange(1, n+1)
    dur = np.sum(times * cfs * df_n) / pv
    durations.append(dur)

durations = np.array(durations)

# -------------------------
# 2. Compute expected returns for each bond
# -------------------------

expected_returns = returns_1y[[f"bond_{n}Y" for n in maturities]].mean().values

# -------------------------
# 3. Plot: Duration vs Expected Return
# -------------------------

fig, ax = plt.subplots(figsize=(10, 6))
plt.scatter(durations, expected_returns, color="cyan", edgecolors="white")

for i, n in enumerate(maturities):
    plt.text(durations[i] + 0.05, expected_returns[i], f"{n}Y", fontsize=10)

plt.title("Duration vs Expected 1-Year Return for Coupon Bonds")
plt.xlabel("Macaulay Duration (years)")
plt.ylabel("Expected 1-Year Return")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

print("Durations:", durations)
print("Expected returns:", expected_returns)